<a href="https://colab.research.google.com/github/Abhiroop17/Deep-Learning-using-Python/blob/main/Estimated_Time_of_Arrival.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Importing libraries to Load and Pre-Process data**

In [ ]:
import pandas as pd
import numpy as np
import random

# **Defining two functions for Generating Coordinates and Calculating Distance**

In [ ]:
# Function to generate random coordinates within a fictional city range
def generate_coordinates(lat_range, long_range):
    return (random.uniform(*lat_range), random.uniform(*long_range))

In [ ]:
# Function to calculate simple Euclidean distance and converting degrees to km
def calculate_distance(coord1, coord2):
    return np.sqrt((coord2[0] - coord1[0])**2 + (coord2[1] - coord1[1])**2) * 111

In [ ]:
# City coordinate ranges around Hyderabad (taking hyd as example)
lat_range = (17.235, 17.535)  # latitude
long_range = (78.3367, 78.6367) #longitude

In [ ]:
# Time of day and day of the week options
times_of_day = pd.date_range("00:00", "23:59", freq="15min").time
days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# **Creating a Simulated Dataset**

In [ ]:
# Generating the dataset
data = []
for i in range(1000):
    start_coord = generate_coordinates(lat_range, long_range)
    end_coord = generate_coordinates(lat_range, long_range)
    distance = calculate_distance(start_coord, end_coord)
    num_turns = random.randint(3, 15)
    num_traffic_lights = random.randint(1, num_turns)
    time_of_day = random.choice(times_of_day)
    day_of_week = random.choice(days_of_week)
    # Historical average speed in kilometer/hour (based on number of turns and traffic lights)
    avg_speed = max(10, random.uniform(20, 50) - 0.5 * num_turns - 0.2 * num_traffic_lights)
     # Simulating current traffic conditions (multiplier between 0.5 for heavy traffic to 1.5 for light traffic)
    traffic_multiplier = random.uniform(0.5, 1.5)
    current_speed = avg_speed * traffic_multiplier
    data.append({
        "Route ID": i + 1,
        "Start Latitude": start_coord[0],
        "Start Longitude": start_coord[1],
        "End Latitude": end_coord[0],
        "End Longitude": end_coord[1],
        "Distance (km)": distance,
        "Number of Turns": num_turns,
        "Number of Traffic Lights": num_traffic_lights,
        "Time of Day": time_of_day,
        "Day of the Week": day_of_week,
        "Historical Average Speed (km/h)": avg_speed,
        "Current Speed (km/h)": current_speed,
    })

In [ ]:
# Create a DataFrame
df = pd.DataFrame(data)
# Display the first five rows of the dataset
df.head()

,Route ID,Start Latitude,Start Longitude,End Latitude,End Longitude,Distance (km),Number of Turns,Number of Traffic Lights,Time of Day,Day of the Week,Historical Average Speed (km/h),Current Speed (km/h)
0,1,17.387208,78.622610,17.513447,78.491171,20.229004,15,4,21:30:00,Thursday,32.969682,25.481731
1,2,17.362815,78.337317,17.428508,78.576468,27.528997,14,12,21:45:00,Saturday,13.276214,14.620518
2,3,17.242503,78.380996,17.395449,78.423302,17.614519,12,2,00:15:00,Saturday,30.273686,15.780898
3,4,17.288060,78.519766,17.492357,78.362260,28.634082,7,3,19:45:00,Sunday,38.704454,20.085490
4,5,17.354702,78.518732,17.243020,78.593778,14.935529,13,9,08:15:00,Tuesday,29.568365,23.298014


# **Spliting the data into Train and Test sets**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Calculate ETA
df['ETA'] = df['Distance (km)'] / df['Current Speed (km/h)']
# Features and target variable
X = df.drop(columns=['ETA'])
y = df['ETA']
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# **Importing Libraries for Feature Engineering, Defining the Model and Evaluation**

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# **Feature Engineering**

In [ ]:
# Define preprocessor
numeric_features = ['Distance (km)', 'Number of Turns', 'Number of Traffic Lights', 'Historical Average Speed (km/h)', 'Current Speed (km/h)']
categorical_features = ['Day of the Week']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)])
# Define the model pipeline
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

# **Training the Model**

In [ ]:
# Train the model
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Distance (km)',
                                                   'Number of Turns',
                                                   'Number of Traffic Lights',
                                                   'Historical Average Speed '
                                                   '(km/h)',
                                                   'Current Speed (km/h)']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Day of the Week'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

# **Model Evaluation**

In [ ]:
# Predict on test set
y_pred = model.predict(X_test)
# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.2f} minutes")

Mean Absolute Error: 0.02 minutes


# **Model Prediction on New Routes**

In [ ]:
import pandas as pd

In [ ]:
# new_data is a DataFrame with the same structure as X_test excluding 'ETA' column.
new_data = pd.DataFrame({
    "Route ID": [1002],
    "Start Latitude": [17.385],
    "Start Longitude": [78.4867],
    "End Latitude": [17.450],
    "End Longitude": [78.5500],
    "Distance (km)": [12],
    "Number of Turns": [7],
    "Number of Traffic Lights": [4],
    "Time of Day": pd.to_datetime("19:00").time(),
    "Day of the Week": ["Sunday"],
    "Historical Average Speed (km/h)": [35],
    "Current Speed (km/h)": [25]
})

# Predict the ETA
predicted_eta = model.predict(new_data)
print(f"Predicted ETA in hours: {predicted_eta[0]:.2f} hrs")
# Convert predicted ETA from hours to minutes
predicted_eta_minutes = predicted_eta[0] * 60
print(f"Predicted ETA in minutes: {predicted_eta_minutes:.2f} mins")

Predicted ETA in hours: 0.50 hrs
Predicted ETA in minutes: 30.05 mins


# **To convert and download the simulated data into a csv file dataset (if required)**

In [ ]:
from google.colab import files
df.to_csv('simulated_traffic_data.csv', index=False)
files.download('simulated_traffic_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>